In [22]:
import pandas as pd
import os
import re
from pathlib import Path

# Step 1: Define the folder path containing the CSV files
folder_path = 'C:/Users/skar/Box/saura_self/Proj - Water tool analysis/data/DMR data_dwd_12192025'
mapping_file = 'C:/Users/skar/Box/saura_self/Proj - Water tool analysis/data/map_DMR_params_survey.xlsx'

# Step 2: Read the mapping file and create a dictionary
print("Reading parameter mapping file...")
mapping_df = pd.read_excel(mapping_file, sheet_name='Sheet1')
param_dict = dict(zip(mapping_df['Survey_parameter_mapping'], mapping_df['parameter_desc']))

# Extract only the values (parameter_desc) for filtering
param_values = list(param_dict.values())
print(f"Loaded {len(param_dict)} parameter mappings")
print(f"Will filter by {len(param_values)} parameters")
print(param_values)

# Step 3: Initialize an empty list to store dataframes
dataframes = []

Reading parameter mapping file...
Loaded 11 parameter mappings
Will filter by 11 parameters
['Whole Effluent Toxicity [WET] - P. promelas', 'Solids, suspended percent removal', 'Nitrogen, Total As N', 'Phosphorus, Total As P', 'BOD, carb-5 day, 20 deg C, percent removal', 'Nitrogen, total Kjeldahl', 'Nitrogen, ammonia in bottom deposits', 'Nitrogen, nitrate total [as N]', 'Nitrogen, nitrite total [as N]', 'CBOD5+NH3-N', 'Solids, total']


In [23]:
# Step 4: Iterate through all CSV files in the folder
print("\nLoading CSV files...")
for filename in os.listdir(folder_path):
    if filename.endswith('.csv'):
        # Extract CWNS ID from filename using regex
        match = re.search(r'CWNS_(\d+)\.csv', filename)
        
        if match:
            cwns_id = match.group(1)
            
            # Read the CSV file
            file_path = os.path.join(folder_path, filename)
            df = pd.read_csv(file_path)
            
            # Add CWNS_ID column at the beginning
            df.insert(0, 'CWNS_ID', cwns_id)
            
            # Append to list
            dataframes.append(df)
            print(f"  Loaded {filename} with CWNS_ID: {cwns_id}")


Loading CSV files...
  Loaded CWNS_11000001001.csv with CWNS_ID: 11000001001
  Loaded CWNS_17000721001.csv with CWNS_ID: 17000721001
  Loaded CWNS_17000721007.csv with CWNS_ID: 17000721007
  Loaded CWNS_17000721009.csv with CWNS_ID: 17000721009
  Loaded CWNS_19000134001.csv with CWNS_ID: 19000134001
  Loaded CWNS_25000025001.csv with CWNS_ID: 25000025001
  Loaded CWNS_25000128001.csv with CWNS_ID: 25000128001
  Loaded CWNS_26000596001.csv with CWNS_ID: 26000596001
  Loaded CWNS_26004005011.csv with CWNS_ID: 26004005011
  Loaded CWNS_29001023001.csv with CWNS_ID: 29001023001
  Loaded CWNS_29001023002.csv with CWNS_ID: 29001023002
  Loaded CWNS_32000011001.csv with CWNS_ID: 32000011001
  Loaded CWNS_34001030001.csv with CWNS_ID: 34001030001


C:\Users\skar\AppData\Local\Temp\ipykernel_9896\167220142.py:13: DtypeWarning: Columns (4,25) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


  Loaded CWNS_34006012001.csv with CWNS_ID: 34006012001
  Loaded CWNS_36008024001.csv with CWNS_ID: 36008024001
  Loaded CWNS_39001666001.csv with CWNS_ID: 39001666001
  Loaded CWNS_39001666002.csv with CWNS_ID: 39001666002
  Loaded CWNS_39001792002.csv with CWNS_ID: 39001792002
  Loaded CWNS_39002093001.csv with CWNS_ID: 39002093001
  Loaded CWNS_47000940002.csv with CWNS_ID: 47000940002
  Loaded CWNS_48000004001.csv with CWNS_ID: 48000004001
  Loaded CWNS_48003033002.csv with CWNS_ID: 48003033002
  Loaded CWNS_48003033005.csv with CWNS_ID: 48003033005
  Loaded CWNS_48004122001.csv with CWNS_ID: 48004122001
  Loaded CWNS_48008015001.csv with CWNS_ID: 48008015001
  Loaded CWNS_51000161001.csv with CWNS_ID: 51000161001


In [24]:
# Step 5: Combine all dataframes into one
if dataframes:
    combined_df = pd.concat(dataframes, ignore_index=True)
    print(f"\nCombined {len(dataframes)} files into single dataframe")
    print(f"Total rows: {len(combined_df)}")
    
    # Step 6: Filter dataset by parameter_desc matching dictionary keys
    print("\nFiltering by parameter mapping values...")
    filtered_df = combined_df[combined_df['parameter_desc'].isin(param_values)].copy()
    
    # Add the keys from the mapping dictionary as a new column
    filtered_df['Survey_parameter_mapping'] = filtered_df['parameter_desc'].map({v: k for k, v in param_dict.items()})

    print(f"Filtered to {len(filtered_df)} rows matching parameter mapping values")
    print(f"Unique CWNS_IDs: {filtered_df['CWNS_ID'].nunique()}")

    # Filter out rows with missing or zero dmr_value_nmbr
    filtered_df = filtered_df[filtered_df['dmr_value_nmbr'].notna() & (filtered_df['dmr_value_nmbr'] != 0)]
    print(f"After removing missing/zero dmr_value_nmbr: {len(filtered_df)} rows")
    print(f"Unique CWNS_IDs after filtering: {filtered_df['CWNS_ID'].nunique()}")
else:
    print("No CSV files found in the folder.")


Combined 26 files into single dataframe
Total rows: 89668

Filtering by parameter mapping values...
Filtered to 1437 rows matching parameter mapping values
Unique CWNS_IDs: 13
After removing missing/zero dmr_value_nmbr: 1216 rows
Unique CWNS_IDs after filtering: 13


In [25]:
# Step 7: Keep only specified columns
"""
columns_to_keep = [
    'CWNS_ID',
    'npdes_id',
    'perm_feature_type_desc',
    'parameter_desc',
    'limit_value_type_desc',
    'limit_value_nmbr',
    'limit_unit_desc',
    'standard_unit_desc',
    'limit_value_standard_units',
    'statistical_base_short_desc',
    'statistical_base_type_code',
    'statistical_base_type_desc',
    'limit_value_qualifier_code',
    'value_type_desc',
    'dmr_value_nmbr',
    'dmr_unit_desc',
    'dmr_value_standard_units',
    'dmr_value_qualifier_code',
    'value_received_date'
]

# Filter to only columns that exist in the dataframe
existing_columns = [col for col in columns_to_keep if col in filtered_df.columns]
filtered_df = filtered_df[existing_columns].copy()
print(f"Kept {len(existing_columns)} columns")
print(f"Total rows: {len(filtered_df)}")
print(f"Unique CWNS_IDs: {filtered_df['CWNS_ID'].nunique()}")
"""

'\ncolumns_to_keep = [\n    \'CWNS_ID\',\n    \'npdes_id\',\n    \'perm_feature_type_desc\',\n    \'parameter_desc\',\n    \'limit_value_type_desc\',\n    \'limit_value_nmbr\',\n    \'limit_unit_desc\',\n    \'standard_unit_desc\',\n    \'limit_value_standard_units\',\n    \'statistical_base_short_desc\',\n    \'statistical_base_type_code\',\n    \'statistical_base_type_desc\',\n    \'limit_value_qualifier_code\',\n    \'value_type_desc\',\n    \'dmr_value_nmbr\',\n    \'dmr_unit_desc\',\n    \'dmr_value_standard_units\',\n    \'dmr_value_qualifier_code\',\n    \'value_received_date\'\n]\n\n# Filter to only columns that exist in the dataframe\nexisting_columns = [col for col in columns_to_keep if col in filtered_df.columns]\nfiltered_df = filtered_df[existing_columns].copy()\nprint(f"Kept {len(existing_columns)} columns")\nprint(f"Total rows: {len(filtered_df)}")\nprint(f"Unique CWNS_IDs: {filtered_df[\'CWNS_ID\'].nunique()}")\n'

In [26]:
# Step 8: Extract year from value_received_date and remove original date column
print("\nProcessing dates...")
filtered_df['value_received_date'] = pd.to_datetime(filtered_df['value_received_date'], errors='coerce')
filtered_df['value_received_year'] = filtered_df['value_received_date'].dt.year
filtered_df = filtered_df.drop('value_received_date', axis=1)
print(f"Unique extracted years {filtered_df['value_received_year'].unique()}")
print(f"Total rows: {len(filtered_df)}")
print(f"Unique CWNS_IDs: {filtered_df['CWNS_ID'].nunique()}")

# save to excel for inspection
output_inspect_file = Path(folder_path) / 'DMR_filtered_inspection.xlsx'
filtered_df.to_excel(output_inspect_file, index=False)


Processing dates...
Unique extracted years [2022 2023 2024 2025]
Total rows: 1216
Unique CWNS_IDs: 13


In [27]:
# Step 9: Average limit_value_nmbr and dmr_value_nmbr by year and other columns as UID
print("\nAggregating data by year...")

# Define aggregation dictionary
agg_dict = {
    'limit_value_nmbr': 'mean',
    'dmr_value_nmbr': 'mean',
    'limit_value_standard_units' : 'mean', 
    'dmr_value_standard_units' : 'mean'
}

groupby_columns = ['CWNS_ID',
                   'value_received_year',
                   'parameter_desc',
                   'Survey_parameter_mapping',
                   'limit_unit_desc',
                   'standard_unit_desc',
                   'limit_value_type_desc',
                   'statistical_base_short_desc',
                   'statistical_base_type_desc',
                   'limit_value_qualifier_code',
                   'value_type_desc',
                   'dmr_unit_desc',
                   'dmr_value_qualifier_code']

# Select only External Outfall and relevant monitoring location types
summarized_df = filtered_df.loc[filtered_df['perm_feature_type_desc'].isin(['External Outfall'])].copy()
summarized_df = summarized_df.loc[summarized_df['monitoring_location_desc'].isin(['Effluent Gross', 'Percent Removal'])].copy()

print(f"Unique CWNS_IDs before summarizing: {summarized_df['CWNS_ID'].nunique()}")

# Aggregate: average the numeric columns
summarized_df = summarized_df.groupby(groupby_columns, as_index=False, dropna=False).agg(agg_dict)
print(f"Summarized to {len(summarized_df)} unique records")

print(f"Unique CWNS_IDs after summarizing: {summarized_df['CWNS_ID'].nunique()}")



Aggregating data by year...
Unique CWNS_IDs before summarizing: 11
Summarized to 117 unique records
Unique CWNS_IDs after summarizing: 11


In [28]:
# Step 10: Save to XLSX file
output_path = os.path.join(folder_path, 'DMR_effluent_data.xlsx')
summarized_df.to_excel(output_path, index=False, sheet_name='Data')
print(f"\nCombined and processed data saved to: {output_path}")
print(f"Final dataset: {len(summarized_df)} rows × {len(summarized_df.columns)} columns")

# Also save the raw combined data to a separate sheet for reference
combined_output_path = os.path.join(folder_path, 'DMR_effluent_data_raw.xlsx')
combined_df.to_excel(combined_output_path, index=False, sheet_name='Raw Data')
print(f"Raw combined data saved to: {combined_output_path}")


Combined and processed data saved to: C:/Users/skar/Box/saura_self/Proj - Water tool analysis/data/DMR data_dwd_12192025\DMR_effluent_data.xlsx
Final dataset: 117 rows × 17 columns
Raw combined data saved to: C:/Users/skar/Box/saura_self/Proj - Water tool analysis/data/DMR data_dwd_12192025\DMR_effluent_data_raw.xlsx
